In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs (timestamps kept as-is per request)
df_metric = pd.read_csv("metric.csv", low_memory=False)
df_log = pd.read_csv("log.csv", low_memory=False)
df_trace = pd.read_csv("trace.csv", low_memory=False)

# Ensure 'value' columns are numeric where present (coerce non-numeric -> NaN)
for df in (df_metric, df_log, df_trace):
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

# --- metric.csv aggregation ---
# Group by (cmdb_id, kpi_name) and compute count, global P95, global max
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_of_points = ('value', 'count'),
        global_P95_of_value = ('value', lambda s: float(np.nanpercentile(s.dropna(), 95)) if s.dropna().size>0 else np.nan),
        global_max_of_value = ('value', 'max')
    )
)

metric_agg_top50 = metric_agg.sort_values('count_of_points', ascending=False).head(50)

# --- log.csv aggregation ---
log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count_of_points = ('value', 'count'),
        global_P95_of_value = ('value', lambda s: float(np.nanpercentile(s.dropna(), 95)) if s.dropna().size>0 else np.nan),
        global_max_of_value = ('value', 'max')
    )
)

log_agg_top50 = log_agg.sort_values('count_of_points', ascending=False).head(50)

# --- trace.csv aggregation ---
trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count_of_points = ('value', 'count'),
        global_P95_of_value = ('value', lambda s: float(np.nanpercentile(s.dropna(), 95)) if s.dropna().size>0 else np.nan),
        global_max_of_value = ('value', 'max')
    )
)

trace_agg_top50 = trace_agg.sort_values('count_of_points', ascending=False).head(50)

# Up to 5 example raw rows from each file to confirm schema (timestamps left as-is)
metric_sample = df_metric.head(5)
log_sample = df_log.head(5)
trace_sample = df_trace.head(5)

# Display compact summaries: aggregated top-50 tables and samples (as required, separated by commas)
metric_agg_top50, metric_sample, log_agg_top50, log_sample, trace_agg_top50, trace_sample

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry aggregates:\n"
    "- Metrics: The top metric series (25 points each) show the 'carts' service with notable KPIs: "
    "cpu (P95 ~1.54, max ~1.60), latency-50 (P95 ~0.0204s), latency-90 (P95 ~0.0453s), "
    "mem (P95 ~2.138e8, max ~2.1386e8), and socket (P95 ~12.12, max ~12.33). "
    "Other components appearing in the top 50 include queue-master (diskio, mem, socket) and rabbitmq (cpu, diskio).\n"
    "- Logs: No widespread errors detected — many services have log.error_count with P95=0 and max=0. "
    "Front-end shows very high row_count (P95 ~2053, max ~2082). Catalogue, user, orders, payment, shipping show moderate row_count values.\n"
    "- Traces: trace.csv contains no records (empty), so no trace-based signals are available.\n\n"
    "Interpretation: There are no elevated application error counts in logs, but metrics point to resource/latency signals in 'carts' (high memory, socket counts and increased latencies). "
    "Front-end has high request volume per logs. With no trace data and no log errors, the most prominent candidates for investigation are resource/latency issues in the 'carts' component.\n"
    "If you want, I can next: (1) show the full top-50 metric table again, (2) inspect time slices for 'carts' metrics, or (3) search error_logs.csv for any textual errors."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count_of_points  global_P95_of_value  global_max_of_value
0          carts         cpu               25         1.542274e+00         1.596418e+00
1          carts  latency-50               25         2.040036e-02         2.051876e-02
2          carts  latency-90               25         4.528130e-02         4.573574e-02
3          carts         mem               25         2.138083e+08         2.138647e+08
4          carts      socket               25         1.212000e+01         1.232727e+01
..           ...         ...              ...                  ...                  ...
45  queue-master      diskio               25         4.337329e+04         4.573590e+04
46  queue-master         mem               25         2.584010e+08         2.584031e+08
47  queue-master      socket               25         3.000000e+00         3.000000e+00
48      rabbitmq         cpu               25         8.330242e-01         8.376612e-01
49      rabbitmq      diskio               25         0.000000e+00         0.000000e+00

[50 rows x 5 columns],     timestamp cmdb_id    kpi_name         value
0  1705766100   carts         cpu  1.020471e+00
1  1705766100   carts  latency-50  2.034240e-02
2  1705766100   carts  latency-90  4.396267e-02
3  1705766100   carts         mem  2.137375e+08
4  1705766100   carts      socket  1.000000e+01,          cmdb_id         log_name  count_of_points  global_P95_of_value  global_max_of_value
7      front-end    log.row_count               25               2053.4                 2082
6      front-end  log.error_count               25                  0.0                    0
5      catalogue    log.row_count               25                169.8                  170
4      catalogue  log.error_count               25                  0.0                    0
12  queue-master  log.error_count               25                  0.0                    0
13  queue-master    log.row_count               25                594.0                  603
17      shipping    log.row_count               25                 66.0                   67
16      shipping  log.error_count               25                  0.0                    0
8         orders  log.error_count               25                  0.0                    0
9         orders    log.row_count               25                131.0                  134
10       payment  log.error_count               25                  0.0                    0
11       payment    log.row_count               25                105.8                  107
18          user  log.error_count               25                  0.0                    0
19          user    log.row_count               25                621.8                  624
2       carts-db  log.error_count               24                  0.0                    0
3       carts-db    log.row_count               24                  1.0                    2
0          carts  log.error_count               22                  0.0                    0
1          carts    log.row_count               22                100.0                  125
15    session-db    log.row_count                5                  5.0                    5
14    session-db  log.error_count                5                  0.0                    0,     timestamp    cmdb_id         log_name  value
0  1705766100  catalogue  log.error_count      0
1  1705766100  catalogue    log.row_count     20
2  1705766100  front-end  log.error_count      0
3  1705766100  front-end    log.row_count    248
4  1705766100     orders  log.error_count      0, Empty DataFrame
Columns: [cmdb_id, trace_name, count_of_points, global_P95_of_value, global_max_of_value]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse previously loaded dataframes if present; otherwise load from CSVs
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv", low_memory=False)
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv", low_memory=False)
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv", low_memory=False)

# Ensure numeric 'value' where applicable
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Parse timestamps to UTC datetimes for filtering (keep original timestamp columns as-is)
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using all metric points (before any filtering)
metric_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_of_points=('value', 'count'),
        global_P95_of_value=('value', lambda s: float(np.nanpercentile(s.dropna(), 95)) if s.dropna().size>0 else np.nan),
        global_max_of_value=('value', 'max')
    )
    .rename(columns={'global_P95_of_value': 'global_P95'})
    .loc[:, ['cmdb_id', 'kpi_name', 'count_of_points', 'global_P95', 'global_max_of_value']]
)

# 2) Define incident window and filter metric.csv to that window (inclusive)
start = pd.to_datetime("2024-01-20T15:52:54Z").tz_convert("UTC")
end = pd.to_datetime("2024-01-20T16:22:54Z").tz_convert("UTC")

df_metric_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()

# 3) For each (cmdb_id, kpi_name) present in window compute counts and anomalies using global_P95 thresholds
# Merge thresholds into windowed data to compare values against global P95
df_metric_window = df_metric_window.merge(
    metric_thresholds[['cmdb_id', 'kpi_name', 'global_P95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Determine anomaly flag where value >= global_P95 (treat NaN global_P95 as no anomaly)
df_metric_window['is_anomaly'] = False
mask_valid = df_metric_window['global_P95'].notna() & df_metric_window['value'].notna()
df_metric_window.loc[mask_valid, 'is_anomaly'] = df_metric_window.loc[mask_valid, 'value'] >= df_metric_window.loc[mask_valid, 'global_P95']

# Aggregate per series in-window
metric_anomalies = (
    df_metric_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_in_window = ('value', 'count'),
        anomaly_count = ('is_anomaly', 'sum'),
        earliest_anomaly_timestamp = ('timestamp', lambda s: int(s[df_metric_window.loc[s.index, 'is_anomaly']].min()) if any(df_metric_window.loc[s.index, 'is_anomaly']) else np.nan),
        max_value_in_window = ('value', 'max'),
        global_P95 = ('global_P95', 'first')
    )
)

# Keep only rows with anomaly_count > 0 and sort
metric_anomalies_with_issues = (
    metric_anomalies[metric_anomalies['anomaly_count'] > 0]
    .sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# 4) Up to 10 example raw metric rows from the incident window (timestamp left as-is)
metric_window_examples = df_metric_window.loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value']].head(10)

# 5) Filter log.csv to same incident window and summarize (cmdb_id, log_name): count_in_window, min(value), max(value)
df_log_window = df_log[(df_log['ts'] >= start) & (df_log['ts'] <= end)].copy()
log_summary_window = (
    df_log_window
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count_in_window = ('value', 'count'),
        min_value = ('value', 'min'),
        max_value = ('value', 'max')
    )
    .sort_values('count_in_window', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
log_window_examples = df_log_window.loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']].head(10)

# 6) Filter error_logs.csv to incident window and return up to 20 rows (timestamp, cmdb_id, message)
df_error_window = df_error[(df_error['ts'] >= start) & (df_error['ts'] <= end)].loc[:, ['timestamp', 'cmdb_id', 'message']]
if df_error_window.shape[0] == 0:
    error_window_examples = "error_logs within window: EMPTY"
else:
    error_window_examples = df_error_window.head(20).reset_index(drop=True)

# Display compact results (variables)
metric_thresholds.head(20), metric_anomalies_with_issues, metric_window_examples, log_summary_window, log_window_examples, error_window_examples

```
Out[2]:
```
Summary of findings (incident window 2024-01-20T15:52:54Z — 2024-01-20T16:22:54Z):

- Global P95 thresholds were computed for every (cmdb_id, kpi_name) from all metric points and used as the anomaly cutoffs.

- Key metric anomalies (only series with anomaly_count > 0 shown in results):
  - Widespread socket metrics are at-or-above their global P95 across many services/databases: rabbitmq, user-db, carts-db, catalogue, orders-db, payment, catalogue-db, queue-master, session-db, etc. Most of these series show anomaly_count = 25 (i.e., all in-window points flagged). Earliest anomaly timestamps for many flagged series equal 1705766100 (the first in-window sample).
  - rabbitmq-exporter memory shows all in-window points >= global P95 (anomaly_count = 25) with global_P95 = max = 4.661248e+06.
  - A few DB and service memory series (catalogue-db, orders, shipping) show a small number of anomalies (anomaly_count = 2).
  - Note: some series whose global_P95 = 0 (e.g., certain "error" metrics) will be counted as anomalies because value >= 0; interpret those cautiously.

- Metric examples from the window confirm presence of carts metrics (cpu, latency-50, latency-90, mem, socket) and carts-db metrics at the window start.

- Logs during the window:
  - No elevated error counts: most cmdb_id log.error_count series show min/max = 0.
  - Front-end shows high row_count (count_in_window = 25, min ~248, max ~2082). Other services show normal/moderate row_count values.
  - Up to 10 example log rows confirm the schema and values.

- error_logs.csv in the window: empty (no error log rows within the incident window).

Interpretation / likely root signals:
- The dominant, consistent signal is elevated socket metrics across many services and DBs during the entire window, plus a significant memory signal for rabbitmq-exporter. Logs do not show application error spikes and error_logs are empty. This pattern points toward socket/connection-related resource stress (or network/socket-level issues) affecting multiple components, with a notable memory spike on rabbitmq-exporter that may be related.

Recommended next steps:
1. Inspect socket/connection counts and limits on the listed components (rabbitmq, DBs, carts-db, user-db, payment, catalogue, queue-master).
2. Check network connectivity and any recent configuration/deployment that could increase connections or exhaust socket resources.
3. Investigate rabbitmq-exporter memory spike (heap/usage) and RabbitMQ connection counts.
4. If desired, I can extract the per-series time slices for the top flagged socket and rabbitmq-exporter mem metrics, or search error_logs more broadly outside this window.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count_of_points    global_P95  global_max_of_value
0          carts         cpu               25  1.542274e+00         1.596418e+00
1          carts  latency-50               25  2.040036e-02         2.051876e-02
2          carts  latency-90               25  4.528130e-02         4.573574e-02
3          carts         mem               25  2.138083e+08         2.138647e+08
4          carts      socket               25  1.212000e+01         1.232727e+01
5          carts    workload               25  8.644900e+00         9.011133e+00
6       carts-db         cpu               25  6.928139e+00         6.999991e+00
7       carts-db      diskio               25  2.392351e+06         2.444671e+06
8       carts-db         mem               25  9.920280e+07         9.928756e+07
9       carts-db      socket               25  7.000000e+00         7.000000e+00
10     catalogue         cpu               25  1.899846e-01         1.950021e-01
11     catalogue  latency-50               25  3.133003e-03         3.163518e-03
12     catalogue  latency-90               25  4.839406e-03         4.894332e-03
13     catalogue         mem               25  6.284616e+06         6.287497e+06
14     catalogue      socket               25  6.000000e+00         6.000000e+00
15     catalogue    workload               25  4.318640e+00         4.329817e+00
16  catalogue-db         cpu               25  2.142436e-01         2.178190e-01
17  catalogue-db      diskio               25  0.000000e+00         0.000000e+00
18  catalogue-db         mem               25  4.306446e+08         4.306660e+08
19  catalogue-db      socket               25  4.000000e+00         4.000000e+00,               cmdb_id kpi_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window    global_P95
0   rabbitmq-exporter      mem               25             25                  1705766100         4.661248e+06  4.661248e+06
1            rabbitmq   socket               25             25                  1705766100         1.100000e+01  1.100000e+01
2             user-db   socket               25             25                  1705766100         8.000000e+00  8.000000e+00
3            carts-db   socket               25             25                  1705766100         7.000000e+00  7.000000e+00
4           catalogue   socket               25             25                  1705766100         6.000000e+00  6.000000e+00
5           orders-db   socket               25             25                  1705766100         5.000000e+00  5.000000e+00
6             payment   socket               25             25                  1705766100         5.000000e+00  5.000000e+00
7        catalogue-db   socket               25             25                  1705766100         4.000000e+00  4.000000e+00
8        queue-master   socket               25             25                  1705766100         3.000000e+00  3.000000e+00
9          session-db   socket               25             25                  1705766100         3.000000e+00  3.000000e+00
10  rabbitmq-exporter   socket               25             25                  1705766100         1.000000e+00  1.000000e+00
11       catalogue-db   diskio               25             25                  1705766100         0.000000e+00  0.000000e+00
12          front-end    error               25             25                  1705766100         0.000000e+00  0.000000e+00
13           rabbitmq   diskio               25             25                  1705766100         0.000000e+00  0.000000e+00
14               user    error               25             25                  1705766100         0.000000e+00  0.000000e+00
15               user   socket               25             20                  1705766100         1.700000e+01  1.700000e+01
16           shipping   socket               25              5                  1705766340         6.000000e+00  6.000000e+00
17       catalogue-db      mem               25              2                  1705766700         4.306660e+08  4.306446e+08
18             orders      mem               25              2                  1705766340         3.332867e+08  3.332657e+08
19           shipping      mem               25              2                  1705767120         2.991251e+08  2.991230e+08,     timestamp   cmdb_id    kpi_name         value
0  1705766100     carts         cpu  1.020471e+00
1  1705766100     carts  latency-50  2.034240e-02
2  1705766100     carts  latency-90  4.396267e-02
3  1705766100     carts         mem  2.137375e+08
4  1705766100     carts      socket  1.000000e+01
5  1705766100     carts    workload  7.744667e+00
6  1705766100  carts-db         cpu  4.869286e+00
7  1705766100  carts-db      diskio  1.286824e+06
8  1705766100  carts-db         mem  9.381888e+07
9  1705766100  carts-db      socket  7.000000e+00,          cmdb_id         log_name  count_in_window  min_value  max_value
0      front-end    log.row_count               25        248       2082
1      front-end  log.error_count               25          0          0
2      catalogue    log.row_count               25         20        170
3      catalogue  log.error_count               25          0          0
4   queue-master  log.error_count               25          0          0
5   queue-master    log.row_count               25         36        603
6       shipping    log.row_count               25          4         67
7       shipping  log.error_count               25          0          0
8         orders  log.error_count               25          0          0
9         orders    log.row_count               25          8        134
10       payment  log.error_count               25          0          0
11       payment    log.row_count               25          8        107
12          user  log.error_count               25          0          0
13          user    log.row_count               25         59        624
14      carts-db  log.error_count               24          0          0
15      carts-db    log.row_count               24          1          2
16         carts  log.error_count               22          0          0
17         carts    log.row_count               22         25        125
18    session-db    log.row_count                5          5          5
19    session-db  log.error_count                5          0          0,     timestamp       cmdb_id         log_name  value
0  1705766100     catalogue  log.error_count      0
1  1705766100     catalogue    log.row_count     20
2  1705766100     front-end  log.error_count      0
3  1705766100     front-end    log.row_count    248
4  1705766100        orders  log.error_count      0
5  1705766100        orders    log.row_count      8
6  1705766100       payment  log.error_count      0
7  1705766100       payment    log.row_count      8
8  1705766100  queue-master  log.error_count      0
9  1705766100  queue-master    log.row_count     36, 'error_logs within window: EMPTY')```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing variables from prior steps: df_metric_window, metric_thresholds, df_metric
# If not present, load/compute minimally (but they should exist per prior runs)
try:
    df_metric_window
except NameError:
    df_metric = pd.read_csv("metric.csv", low_memory=False)
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    # recompute thresholds (global P95) if missing
    metric_thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(global_P95=('value', lambda s: float(np.nanpercentile(s.dropna(), 95)) if s.dropna().size>0 else np.nan))
    )
    # parse timestamps and filter window
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
    start = pd.to_datetime("2024-01-20T15:52:54Z").tz_convert("UTC")
    end = pd.to_datetime("2024-01-20T16:22:54Z").tz_convert("UTC")
    df_metric_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()
    df_metric_window = df_metric_window.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Ensure required columns exist
if 'global_P95' not in df_metric_window.columns:
    # merge thresholds
    metric_thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(global_P95=('value', lambda s: float(np.nanpercentile(s.dropna(), 95)) if s.dropna().size>0 else np.nan))
    )
    df_metric_window = df_metric_window.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Work on a copy with necessary columns
dfw = df_metric_window[['timestamp', 'cmdb_id', 'kpi_name', 'value', 'global_P95']].copy()
dfw['timestamp'] = pd.to_numeric(dfw['timestamp'], errors='coerce').astype('Int64')  # keep UNIX seconds as ints

# 1) Identify rows where value >= global_P95 AND global_P95 > 0
mask = dfw['global_P95'].notna() & (dfw['global_P95'] > 0) & dfw['value'].notna() & (dfw['value'] >= dfw['global_P95'])
df_anom = dfw[mask].copy()

# If no anomalies, produce empty outputs
if df_anom.shape[0] == 0:
    segments_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
        'length_in_points','max_value_in_segment','global_P95','severity_ratio'
    ])
    earliest_segment = pd.DataFrame(columns=['cmdb_id','kpi_name','segment_start_timestamp'])
    example_rows_1 = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
    example_rows_2 = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
else:
    # 1) & 2) Detect consecutive segments per group. Consider consecutive if timestamp diff == 60 seconds.
    df_anom = df_anom.sort_values(['cmdb_id', 'kpi_name', 'timestamp']).reset_index(drop=True)
    # compute time diff within each group
    df_anom['time_diff'] = df_anom.groupby(['cmdb_id', 'kpi_name'])['timestamp'].diff().fillna(9999).astype('Int64')
    # new segment when time_diff != 60
    df_anom['new_segment'] = (df_anom['time_diff'] != 60)
    # assign segment ids per group
    df_anom['segment_id'] = df_anom.groupby(['cmdb_id', 'kpi_name'])['new_segment'].cumsum()
    # aggregate segments
    segs = (
        df_anom
        .groupby(['cmdb_id', 'kpi_name', 'segment_id'], as_index=False)
        .agg(
            segment_start_timestamp = ('timestamp', 'min'),
            segment_end_timestamp = ('timestamp', 'max'),
            length_in_points = ('timestamp', 'count'),
            max_value_in_segment = ('value', 'max'),
            global_P95 = ('global_P95', 'first')
        )
    )
    # 2) Exclude segments shorter than 3 points
    segs = segs[segs['length_in_points'] >= 3].copy()
    if segs.shape[0] == 0:
        segments_top20 = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','segment_start_timestamp','segment_end_timestamp',
            'length_in_points','max_value_in_segment','global_P95','severity_ratio'
        ])
        earliest_segment = pd.DataFrame(columns=['cmdb_id','kpi_name','segment_start_timestamp'])
        example_rows_1 = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
        example_rows_2 = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
    else:
        # 1) compute severity_ratio
        segs['severity_ratio'] = (segs['max_value_in_segment'] - segs['global_P95']) / segs['global_P95']
        # 3) Sort by severity_ratio desc then length desc, return top 20
        segments_top20 = segs.sort_values(['severity_ratio', 'length_in_points'], ascending=[False, False]).head(20).reset_index(drop=True)
        # 4) earliest segment_start_timestamp across all detected segments
        earliest_idx = segments_top20['segment_start_timestamp'].idxmin()
        earliest_segment = segments_top20.loc[[earliest_idx], ['cmdb_id', 'kpi_name', 'segment_start_timestamp']].reset_index(drop=True)
        # 5) For the two highest-severity segments, return up to 10 example raw metric rows from the incident window
        top_segments = segments_top20.sort_values(['severity_ratio', 'length_in_points'], ascending=[False, False]).head(2)
        example_rows_list = []
        for _, row in top_segments.iterrows():
            cms = row['cmdb_id']
            kpi = row['kpi_name']
            sel = df_metric_window[(df_metric_window['cmdb_id'] == cms) & (df_metric_window['kpi_name'] == kpi)].loc[:, ['timestamp','cmdb_id','kpi_name','value']].sort_values('timestamp').head(10)
            example_rows_list.append(sel.reset_index(drop=True))
        # Provide examples for up to two segments; if only one, fill second with empty DF
        if len(example_rows_list) >= 1:
            example_rows_1 = example_rows_list[0]
        else:
            example_rows_1 = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
        if len(example_rows_list) >= 2:
            example_rows_2 = example_rows_list[1]
        else:
            example_rows_2 = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])

# Final compact outputs
segments_top20, earliest_segment, example_rows_1, example_rows_2

```
Out[3]:
```
Summary of fault-identification within the incident window (2024-01-20T15:52:54Z — 2024-01-20T16:22:54Z UTC):

- What was found:
  - Detected 12 consecutive anomaly segments (each >= 3 points) across multiple services/DBs.
  - Most segments span the full analyzed window (length_in_points = 25); one (user socket) is length 18.
  - The affected KPIs are predominantly socket metrics on many components (carts-db, catalogue, catalogue-db, orders-db, payment, queue-master, rabbitmq, session-db, user-db, user) and rabbitmq-exporter memory (mem) and socket.
  - All detected segments have severity_ratio = 0.0 (max_value_in_segment == global_P95). In other words: values were at the global P95 threshold (sustained high), but not exceeding it.

- Representative top segments (sorted by severity then length):
  - carts-db / socket — segment_start_timestamp: 1705766100, segment_end_timestamp: 1705767540, length: 25, max_value: 7.0, global_P95: 7.0, severity_ratio: 0.0
  - catalogue / socket — segment_start_timestamp: 1705766100, segment_end_timestamp: 1705767540, length: 25, max_value: 6.0, global_P95: 6.0, severity_ratio: 0.0
  - (Other similar full-window socket segments: catalogue-db, orders-db, payment, queue-master, rabbitmq, session-db, user-db, etc.; rabbitmq-exporter mem also spanned the window with max = global_P95 = 4,661,248.)

- Earliest detected segment onset:
  - carts-db / socket at Unix timestamp 1705766100 (the earliest segment_start among detected segments).

- Example raw rows (validation) for the two highest-severity segments:
  - carts-db / socket (first 10 points): repeated entries with value = 7.0 at timestamps beginning 1705766100, 1705766160, 1705766220, ...
  - catalogue / socket (first 10 points): repeated entries with value = 6.0 at timestamps beginning 1705766100, 1705766160, 1705766220, ...

- Interpretation and next steps:
  - There is a widespread, sustained elevation of socket metrics (and a sustained high memory value on rabbitmq-exporter) across many components during the window. Although values do not exceed the global P95 (severity_ratio = 0), their persistence at the P95 indicates sustained high load or utilization.
  - Recommended investigation: check connection/socket utilization and limits, inspect RabbitMQ memory usage and connection counts, review recent deployment/config/network changes that could increase sustained socket usage.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  segment_id  segment_start_timestamp  segment_end_timestamp  length_in_points  max_value_in_segment  global_P95  severity_ratio
0            carts-db   socket           1               1705766100             1705767540                25                   7.0         7.0             0.0
1           catalogue   socket           1               1705766100             1705767540                25                   6.0         6.0             0.0
2        catalogue-db   socket           1               1705766100             1705767540                25                   4.0         4.0             0.0
3           orders-db   socket           1               1705766100             1705767540                25                   5.0         5.0             0.0
4             payment   socket           1               1705766100             1705767540                25                   5.0         5.0             0.0
5        queue-master   socket           1               1705766100             1705767540                25                   3.0         3.0             0.0
6            rabbitmq   socket           1               1705766100             1705767540                25                  11.0        11.0             0.0
7   rabbitmq-exporter      mem           1               1705766100             1705767540                25             4661248.0   4661248.0             0.0
8   rabbitmq-exporter   socket           1               1705766100             1705767540                25                   1.0         1.0             0.0
9          session-db   socket           1               1705766100             1705767540                25                   3.0         3.0             0.0
10            user-db   socket           1               1705766100             1705767540                25                   8.0         8.0             0.0
11               user   socket           1               1705766100             1705767120                18                  17.0        17.0             0.0,     cmdb_id kpi_name  segment_start_timestamp
0  carts-db   socket               1705766100,     timestamp   cmdb_id kpi_name  value
0  1705766100  carts-db   socket    7.0
1  1705766160  carts-db   socket    7.0
2  1705766220  carts-db   socket    7.0
3  1705766280  carts-db   socket    7.0
4  1705766340  carts-db   socket    7.0
5  1705766400  carts-db   socket    7.0
6  1705766460  carts-db   socket    7.0
7  1705766520  carts-db   socket    7.0
8  1705766580  carts-db   socket    7.0
9  1705766640  carts-db   socket    7.0,     timestamp    cmdb_id kpi_name  value
0  1705766100  catalogue   socket    6.0
1  1705766160  catalogue   socket    6.0
2  1705766220  catalogue   socket    6.0
3  1705766280  catalogue   socket    6.0
4  1705766340  catalogue   socket    6.0
5  1705766400  catalogue   socket    6.0
6  1705766460  catalogue   socket    6.0
7  1705766520  catalogue   socket    6.0
8  1705766580  catalogue   socket    6.0
9  1705766640  catalogue   socket    6.0)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv", low_memory=False)
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Ensure numeric value and timestamp-parsed column exist
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Compute global P90 thresholds using the entire metric series (before filtering)
metric_p90 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(global_P90 = ('value', lambda s: float(np.nanpercentile(s.dropna(), 90)) if s.dropna().size>0 else np.nan))
)

# 2) Filter metric.csv to the incident window (UTC)
start = pd.to_datetime("2024-01-20T15:52:54Z").tz_convert("UTC")
end = pd.to_datetime("2024-01-20T16:22:54Z").tz_convert("UTC")
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()

# Restrict to candidate services
candidates = ["carts", "catalogue", "orders", "payment", "user"]
df_candidates = df_window[df_window['cmdb_id'].isin(candidates)].copy()

# Merge global P90 into the windowed candidate dataframe
df_candidates = df_candidates.merge(metric_p90, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Aggregate per (cmdb_id, kpi_name) within candidates
# Determine anomaly flag: value >= global_P90 (treat NaN global_P90 as False)
mask_valid = df_candidates['global_P90'].notna() & df_candidates['value'].notna()
df_candidates['is_anomaly'] = False
df_candidates.loc[mask_valid, 'is_anomaly'] = df_candidates.loc[mask_valid, 'value'] >= df_candidates.loc[mask_valid, 'global_P90']

agg = (
    df_candidates
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_in_window = ('value', 'count'),
        anomaly_count = ('is_anomaly', 'sum'),
        earliest_anomaly_timestamp = ('timestamp', lambda s: int(s[df_candidates.loc[s.index, 'is_anomaly']].min()) if any(df_candidates.loc[s.index, 'is_anomaly']) else np.nan),
        max_value_in_window = ('value', 'max'),
        global_P90 = ('global_P90', 'first')
    )
)

# Only include rows with anomaly_count > 0
agg = agg[agg['anomaly_count'] > 0].copy()

# Compute severity_ratio safely (avoid division by zero)
agg['severity_ratio'] = np.nan
mask_p90_pos = agg['global_P90'].notna() & (agg['global_P90'] != 0)
agg.loc[mask_p90_pos, 'severity_ratio'] = (agg.loc[mask_p90_pos, 'max_value_in_window'] - agg.loc[mask_p90_pos, 'global_P90']) / agg.loc[mask_p90_pos, 'global_P90']

# Sort by severity_ratio desc (NaNs last), then anomaly_count desc
agg_sorted = agg.sort_values(['severity_ratio', 'anomaly_count'], ascending=[False, False], na_position='last').reset_index(drop=True)

# Limit to top 20 and select required columns
results_top20 = agg_sorted.loc[:, ['cmdb_id', 'kpi_name', 'count_in_window', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'global_P90', 'severity_ratio']].head(20)

# 4) For the top 2 (cmdb_id,kpi_name), return up to 10 example raw metric rows from the incident window
example_rows_1 = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
example_rows_2 = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])
top_pairs = results_top20[['cmdb_id','kpi_name']].head(2)

for idx, row in top_pairs.reset_index(drop=True).iterrows():
    cms = row['cmdb_id']
    kpi = row['kpi_name']
    sel = df_candidates[(df_candidates['cmdb_id']==cms) & (df_candidates['kpi_name']==kpi)].loc[:, ['timestamp','cmdb_id','kpi_name','value']].sort_values('timestamp').head(10).reset_index(drop=True)
    if idx == 0:
        example_rows_1 = sel
    elif idx == 1:
        example_rows_2 = sel

# Final compact outputs
results_top20, example_rows_1, example_rows_2

```
Out[4]:
```
Summary of results (candidate services: carts, catalogue, orders, payment, user; anomaly = value >= global P90):

- Method: global P90 computed from the full metric series per (cmdb_id,kpi_name). Metrics filtered to the incident window 2024-01-20T15:52:54Z—2024-01-20T16:22:54Z. Only series with anomaly_count > 0 are reported; severity_ratio = (max_value_in_window - global_P90)/global_P90.

- Top findings (sorted by severity_ratio):
  1. orders / cpu — count_in_window: 25, anomaly_count: 3, earliest_anomaly_timestamp: 1705766100, max_value_in_window: 7.214450, global_P90: 2.541856, severity_ratio ≈ 1.838
  2. payment / cpu — count_in_window: 25, anomaly_count: 3, earliest_anomaly_timestamp: 1705766100, max_value_in_window: 0.274815, global_P90: 0.105034, severity_ratio ≈ 1.616
  3. payment / latency-50 — anomaly_count: 3, max 0.092680 vs P90 0.058117, severity_ratio ≈ 0.595
  4. carts / workload, catalogue / cpu, carts / cpu, orders / workload, carts / socket, payment / workload, carts / latency-90, user / latency-50, user / cpu, user / workload, orders / socket, orders / latency-90, catalogue latencies, catalogue / workload, carts / latency-50 — (these appear with smaller severity_ratios; most anomaly_count = 3, one carts/socket anomaly_count = 4).
  - In total the top-20 table returned 20 (cmdb_id,kpi_name) rows meeting anomaly_count>0.

- Pattern notes:
  - The highest-severity signals are short CPU bursts in orders and payment (anomaly_count = 3 points within the 25-point window), with orders.cpu reaching 7.21445 (≈ 184% above its P90).
  - Most other anomalies are modest (severity_ratio << 1) and typically involve workload, latency, socket, or CPU metrics with anomaly_count = 3 (some 4).
  - These are concentrated in the candidate services rather than being widespread across unrelated services.

- Example raw rows for top 2 pairs (first up to 10 rows each, from the incident window) — useful to validate shapes:
  - orders / cpu (first 10 in-window rows): timestamps 1705766100..1705766640 with values [2.781320, 5.153760, 1.420877, 7.214450, 1.119664, 1.190431, 1.169343, 1.174506, 1.101279, 1.159349]
    - Note: multiple values exceed P90 (2.541856), including a large spike 7.214450.
  - payment / cpu (first 10 in-window rows): timestamps 1705766100..1705766640 with values [0.177948, 0.274815, 0.090472, 0.092342, 0.093314, 0.090067, 0.094638, 0.084103, 0.092927, 0.089862]
    - Several values exceed P90 (0.105034), including a short burst with max 0.274815.

- Interpretation / recommended next steps:
  - Investigate short CPU bursts in orders and payment around the listed timestamps (convert UNIX seconds to human time as needed). These have the highest severity ratios and could explain transient performance issues.
  - Correlate these CPU spikes with request traffic, logs, or deployments at the same timestamps. Also check payment latency-50 (elevated) since it shows notable severity.
  - For lower-severity anomalies, monitor trends (they may be sustained near P90) but prioritize the orders/payment CPU events for immediate investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  global_P90  severity_ratio
0      orders         cpu               25              3                  1705766100             7.214450    2.541856        1.838261
1     payment         cpu               25              3                  1705766100             0.274815    0.105034        1.616442
2     payment  latency-50               25              3                  1705767000             0.092680    0.058117        0.594719
3       carts    workload               25              3                  1705766220             9.011133    8.506267        0.059352
4   catalogue         cpu               25              3                  1705766640             0.195002    0.187666        0.039094
5       carts         cpu               25              3                  1705766760             1.596418    1.537820        0.038105
6      orders    workload               25              3                  1705766280             2.239982    2.176207        0.029306
7       carts      socket               25              4                  1705766760            12.327273   12.000000        0.027273
8     payment    workload               25              3                  1705766280             2.232764    2.178320        0.024993
9       carts  latency-90               25              3                  1705766940             0.045736    0.044691        0.023378
10       user  latency-50               25              3                  1705766760             0.005174    0.005056        0.023339
11       user         cpu               25              3                  1705766220             1.104779    1.080838        0.022151
12       user    workload               25              3                  1705766220            19.714600   19.340507        0.019342
13     orders      socket               25              3                  1705767240            16.566667   16.270303        0.018215
14     orders  latency-90               25              3                  1705767180             0.240486    0.236457        0.017038
15  catalogue  latency-90               25              3                  1705766460             0.004894    0.004820        0.015429
16  catalogue  latency-50               25              3                  1705766460             0.003164    0.003122        0.013233
17     orders  latency-50               25              3                  1705767180             0.178048    0.175809        0.012731
18  catalogue    workload               25              3                  1705766220             4.329817    4.282800        0.010978
19      carts  latency-50               25              3                  1705766100             0.020519    0.020341        0.008723,     timestamp cmdb_id kpi_name     value
0  1705766100  orders      cpu  2.781320
1  1705766160  orders      cpu  5.153760
2  1705766220  orders      cpu  1.420877
3  1705766280  orders      cpu  7.214450
4  1705766340  orders      cpu  1.119664
5  1705766400  orders      cpu  1.190431
6  1705766460  orders      cpu  1.169343
7  1705766520  orders      cpu  1.174506
8  1705766580  orders      cpu  1.101279
9  1705766640  orders      cpu  1.159349,     timestamp  cmdb_id kpi_name     value
0  1705766100  payment      cpu  0.177948
1  1705766160  payment      cpu  0.274815
2  1705766220  payment      cpu  0.090472
3  1705766280  payment      cpu  0.092342
4  1705766340  payment      cpu  0.093314
5  1705766400  payment      cpu  0.090067
6  1705766460  payment      cpu  0.094638
7  1705766520  payment      cpu  0.084103
8  1705766580  payment      cpu  0.092927
9  1705766640  payment      cpu  0.089862)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv", low_memory=False)
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv", low_memory=False)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv", low_memory=False)

# Ensure numeric values and parsed timestamps for filtering
if 'value' in df_log.columns:
    df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
if 'value' in df_metric.columns:
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
if 'ts' not in df_error.columns:
    df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Incident window
start = pd.to_datetime("2024-01-20T15:52:54Z").tz_convert("UTC")
end = pd.to_datetime("2024-01-20T16:22:54Z").tz_convert("UTC")

# 1) Filter log.csv to window for orders and payment and aggregate
targets = ["orders", "payment"]
df_log_window = df_log[(df_log['ts'] >= start) & (df_log['ts'] <= end) & (df_log['cmdb_id'].isin(targets))].copy()

log_summary = (
    df_log_window
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count_in_window = ('value', 'count'),
        min_value = ('value', 'min'),
        max_value = ('value', 'max')
    )
    .sort_values('count_in_window', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 2) error_logs.csv rows for orders and payment in window
df_error_window_targets = df_error[(df_error['ts'] >= start) & (df_error['ts'] <= end) & (df_error['cmdb_id'].isin(targets))].loc[:, ['timestamp', 'cmdb_id', 'message']].reset_index(drop=True)
if df_error_window_targets.shape[0] == 0:
    error_rows_result = "error_logs within window for orders/payment: EMPTY"
else:
    error_rows_result = df_error_window_targets.head(20)

# 3) orders.cpu context rows: find in-window anomaly spike timestamps using global P90, then include ±2 samples from full metric series
# Ensure we have global P90 computed; if not, compute now globally
if 'metric_p90' not in globals():
    metric_p90 = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(global_P90 = ('value', lambda s: float(np.nanpercentile(s.dropna(), 90)) if s.dropna().size>0 else np.nan))
    )

# Get orders.cpu full series
df_orders_cpu = df_metric[(df_metric['cmdb_id'] == 'orders') & (df_metric['kpi_name'] == 'cpu')].sort_values('timestamp').reset_index(drop=True)
# Merge global P90 for orders.cpu
p90_row = metric_p90[(metric_p90['cmdb_id']=='orders') & (metric_p90['kpi_name']=='cpu')]
if p90_row.shape[0] == 0:
    orders_cpu_p90 = np.nan
else:
    orders_cpu_p90 = float(p90_row['global_P90'].iloc[0])

# In-window orders.cpu anomaly points (value >= global_P90)
df_orders_cpu['ts'] = pd.to_datetime(df_orders_cpu['timestamp'], unit='s', utc=True)
df_orders_cpu_window = df_orders_cpu[(df_orders_cpu['ts'] >= start) & (df_orders_cpu['ts'] <= end)].copy()

if np.isnan(orders_cpu_p90):
    spike_timestamps = []
else:
    mask_anom = df_orders_cpu_window['value'].notna() & (df_orders_cpu_window['value'] >= orders_cpu_p90)
    df_orders_cpu_window_anom = df_orders_cpu_window[mask_anom].copy()
    # pick top 2 spike timestamps by value (descending), unique timestamps
    spike_timestamps = df_orders_cpu_window_anom.sort_values('value', ascending=False)['timestamp'].unique().tolist()[:2]

# For each spike timestamp, find its position in full df_orders_cpu and take ±2 samples
context_frames = []
for ts_spike in spike_timestamps:
    # find first matching index in full df_orders_cpu
    matches = df_orders_cpu[df_orders_cpu['timestamp'] == ts_spike].index
    if len(matches) == 0:
        continue
    pos = int(matches[0])
    start_idx = max(0, pos - 2)
    end_idx = min(len(df_orders_cpu) - 1, pos + 2)
    ctx = df_orders_cpu.loc[start_idx:end_idx, ['timestamp', 'cmdb_id', 'kpi_name', 'value']].reset_index(drop=True)
    context_frames.append(ctx)

if len(context_frames) == 0:
    orders_cpu_context = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'kpi_name', 'value'])
else:
    orders_cpu_context = pd.concat(context_frames, ignore_index=True).drop_duplicates().sort_values('timestamp').reset_index(drop=True).head(10)

# Final compact outputs
log_summary, error_rows_result, orders_cpu_context

```
Out[5]:
```
Summary (incident window 2024-01-20T15:52:54Z — 2024-01-20T16:22:54Z UTC):

1) Logs (orders, payment)
- orders:
  - log.error_count: count_in_window=25, min=0, max=0 (no error count activity).
  - log.row_count: count_in_window=25, min=8, max=134 (request volume varied up to 134).
- payment:
  - log.error_count: count_in_window=25, min=0, max=0 (no error count activity).
  - log.row_count: count_in_window=25, min=8, max=107 (request volume up to 107).

2) error_logs for orders/payment in the window
- None — error_logs.csv returned no rows for orders or payment in the incident window.

3) orders.cpu context around detected spikes
- The largest detected orders.cpu spike: timestamp 1705766280 with value 7.214450 (this is inside the incident window).
- Example metric rows (orders.cpu) around spikes (selected rows returned):
  - (1705766100) 2.781320
  - (1705766160) 5.153760
  - (1705766220) 1.420877
  - (1705766280) 7.214450  <-- largest spike
  - (1705766340) 1.119664
  - (1705766400) 1.190431

Interpretation / suggested next steps
- No log error counts or error log entries for orders/payment during the window, so application errors are not evident in logs.
- There is a short, high CPU spike on orders (7.21445) at unix ts 1705766280 with surrounding lower values. Payment also had earlier anomalies (per previous analyses), but logs do not show errors.
- Recommended actions: inspect orders service activity at the spike time (traffic, request patterns, deployments), check any trace/span data for slow or noisy requests around 1705766280, and correlate with infrastructure (CPU, container host) and external dependencies.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id         log_name  count_in_window  min_value  max_value
0   orders  log.error_count               25          0          0
1   orders    log.row_count               25          8        134
2  payment  log.error_count               25          0          0
3  payment    log.row_count               25          8        107, 'error_logs within window for orders/payment: EMPTY',     timestamp cmdb_id kpi_name     value
0  1705766100  orders      cpu  2.781320
1  1705766160  orders      cpu  5.153760
2  1705766220  orders      cpu  1.420877
3  1705766280  orders      cpu  7.214450
4  1705766340  orders      cpu  1.119664
5  1705766400  orders      cpu  1.190431)```
```